In [1]:
from pathlib import Path
from datetime import date

import plotly.express as px
import polars as pl

In [2]:
collected_vehicle_data_path = Path("..", "raw_data", "collected-data", "collected_vehicle_data.csv")

# Electric Vehicles

## Results

In [3]:
obs = pl.scan_csv(
    source=collected_vehicle_data_path,
    has_header=True
)

In [4]:
n = (obs
    .select(pl.len())  
    .collect() 
    .item()
)

num_elec = (obs
    .filter(pl.col("engine") == "ELECTRIC")
    .select(pl.len())  
    .collect() 
    .item()
)

There were `{python} num_elec` electric vehicles in the sample of `{python} n` vehicles.  Based on this lower number of electric vehicles in the sample than expected, we think it best to extrapolate the registration data itself using a constant function to form our estimate for March 2025.

In [5]:
elec_regs = pl.LazyFrame(
    data=dict(
        cutoff_date=pl.Series([date(2023, 2, 15), date(2024, 2, 15), date(2025, 2, 17)]),
        p=pl.Series([5601/488967, 8414/507164, 12153/525078])
    )
)

In [ ]:
#| warning: false
#| label: fig-elec
#| fig-cap: "Values for the first three dates are calculated by dividing the number of all-electric vehicle registrations by the total number of registrations at the given date cutoff.  The value for the last date is a constant extrapolation." 
x = elec_regs.select("cutoff_date").collect().to_series()
x.extend(pl.Series([date(2025, 3, 15)]))
y = elec_regs.select("p").collect().to_series()
y.extend(y.tail(1))
fig = px.line(
    x=x, 
    y=y, 
    markers="lines+markers"
)
fig.update_layout(
    title="Estimated Relative Frequency of Electric Vehicles in the Target Population",
    xaxis=dict(title="Cutoff Date"),
    yaxis=dict(title="Relative Frequency")
)

/home/justin/bin/mambaforge/envs/justins_room/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:105: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



**Caveat:** No attempts are made to account for the fact that the government registration records are not a complete census of the target population.  The estimate that `{python} f"{round(12153/525078 * 100, 2)}%"` of vehicles in the target population are fully electric is only a rough guess.